In [ ]:
import numpy as np
import pandas as pd

# ============================================================================
# [코드 리뷰 수정 이력 — 한현석, 2026-08-25]
#   1) find_optimal_degree: 학습 데이터로만 R^2를 재서 차수를 고르던 방식을
#      학습/검증 분리 + 검증 R^2 기반 조기종료로 변경 (과적합 방지,
#      target_adj_r2=1.0 하드코딩 제거로 커널 크래시 유발 가능성 있던
#      고차수 폭주 방지)
#   2) 데이터 생성부: 정의역(domain_min/domain_max)을 기억해두었다가
#      theoretical_scale로 스케일링하도록 변경 (min_max_scale은 표본의
#      경험적 min/max라 검증 단계에서 학습 표본 범위를 벗어나면 르장드르
#      기저가 [-1,1] 밖에서 불안정해질 수 있음)
#   3) get_sobol: 정규직교 기저 가정이 이 노트북의 (비정규화) 르장드르
#      기저와 맞지 않아 그대로 쓰면 틀린 값이 나옴 — 경고 주석만 추가,
#      실제 사용은 계속 get_sobol2
#   그 외 로직(basis, PDD 절단 방식, get_sobol2)은 원래 구현이 맞다고
#   판단해 손대지 않음.
# ============================================================================

input = []
domain_min = []  # [추가] 변수별 실제 정의역 하한 (theoretical_scale에 사용)
domain_max = []  # [추가] 변수별 실제 정의역 상한
for _ in range(10):
    x = np.random.randint(1, 10)
    y = np.random.randint(1, 10)
    # [수정] x==y로 뽑히면 uniform(x,y,...)의 폭이 0이 되는 문제를 방지하려고
    #        min/max로 정리. (원래는 x,y를 순서 보장 없이 그대로 넘겼음)
    lo, hi = min(x, y), max(x, y)
    if lo == hi:
        hi += 1
    z = 5000 #시행 횟수
    input.append(np.array(np.random.uniform(lo, hi, z)))
    domain_min.append(lo)
    domain_max.append(hi)

input = np.array(input)
domain_min = np.array(domain_min).reshape(-1, 1)
domain_max = np.array(domain_max).reshape(-1, 1)

'''
# 데이터 불러오기 (구분자가 공백인 경우)
data = np.loadtxt('filename.txt')

# 데이터 불러오기 (구분자가 콤마 ','인 CSV 형태인 경우)
# data = np.loadtxt('filename.txt', delimiter=',')

print(data.shape) # (n, m) 확인 용도

input = np.array(data)
'''

def standard(x):

    x_mean = x.mean(axis=1, keepdims=True)
    x_std = x.std(axis=1, keepdims=True)

    X = (x - x_mean) / x_std
    return X

def min_max_scale(x, min_val=-1, max_val=1):
    # [참고] 이 함수는 표본의 "경험적" min/max로 스케일링함.
    # 변수의 실제 정의역(설계변수 LB/UB 등)을 모를 때만 fallback으로 쓰고,
    # 정의역을 알고 있다면 아래 theoretical_scale을 쓰는 게 안전함
    # (검증/최적화 단계에서 학습 표본 범위를 벗어난 값이 들어오면
    #  르장드르 기저가 [-1,1] 밖에서 값이 급격히 커질 수 있기 때문).
    x_min = x.min(axis=1, keepdims=True)
    x_max = x.max(axis=1, keepdims=True)
    
    range_mask = (x_max - x_min) == 0
    x_max[range_mask] += 1e-8 
    
    x_std = (x - x_min) / (x_max - x_min)
    scaled_x = x_std * (max_val - min_val) + min_val
    return scaled_x

def theoretical_scale(x, domain_min, domain_max, target_min=-1, target_max=1):
    
    x_std = (x - domain_min) / (domain_max - domain_min)

    scaled_x = x_std * (target_max - target_min) + target_min
    
    return scaled_x

# [수정] min_max_scale(input) -> theoretical_scale(input, ...)
# 이유: 위 changelog 2) 참고 — 생성에 쓴 실제 정의역(domain_min/domain_max)으로
#       스케일링해야 학습/검증 단계에서 스케일이 어긋나지 않음.
input = theoretical_scale(input, domain_min=domain_min, domain_max=domain_max)
print(input.shape)

output = np.random.uniform(1, 10, 5000)

print(output.shape)

# 르장드르 기저 형성
def basis(x, a):
    # 0차
    if a == 0:
        return np.ones_like(x)
    # 1차
    if a == 1:
        return x
    
    # 점화식을 위한 초기값 설정
    p_prev2 = np.ones_like(x) # P_{n-1} (처음엔 0차)
    p_prev1 = x.copy()         # P_n     (처음엔 1차)
    p_n = None
    
    for n in range(1, a):
        # 점화식: P_{n+1} = ((2n+1)x*P_n - n*P_{n-1}) / (n+1)
        p_n = ((2 * n + 1) * x * p_prev1 - n * p_prev2) / (n + 1)
        
        p_prev2 = p_prev1
        p_prev1 = p_n
        
    return p_n

def PDD(x,n,y): #x는 입력행렬, n은 다항식 고차항의 최대 차수
    dim = x.shape[0] ## 변수의 개수
    N = x.shape[1] ## 입력값의 개수

    phi = []
    mapping_list = []

    phi.append(np.ones(N)) #상수항
    mapping_list.append([0] * dim)

    for i in range(dim): #1변수에 대해 3차까지
        for j in range(1,n+1):
            phi.append(basis(x[i, :], j)) 
            mapping = [0] * dim
            mapping[i] = 1 # i번째 변수와 관련 있음 표시
            mapping_list.append(mapping)

    if y >= 2:
        for i in range(2,n+1): #2변수 간의 상호작용 분석 n은 현재 고차항의 차수
            for j in range(1,i): #르장드르 다항식에 적용할 차수 / j랑 i-j를 합치면 현재 차수가 되도록 코드 구성
                for k in range(dim):
                    for l in range(k+1,dim):
                        x_a = basis(x[k, :], j)*basis(x[l, :], i-j)
                        phi.append(x_a)
                        mapping = [0] * dim
                        mapping[k] = 1 
                        mapping[l] = 1 
                        mapping_list.append(mapping)

    # 3변수 간의 상호작용 분석 (i: 전체 차수, j: 첫 번째 변수 차수, k: 두 번째 변수 차수)
    if y >= 3:
        for i in range(3, n + 1):  # 전체 차수 i는 최소 3부터 시작 (1+1+1)
            for j in range(1, i - 1):  # 첫 번째 변수의 차수
                for k in range(1, i - j):  # 두 번째 변수의 차수 (남은 차수 i-j 내에서 할당)
                    m = i - j - k  # 세 번째 변수의 차수
                    for v1 in range(dim):  # 첫 번째 변수 인덱스
                        for v2 in range(v1 + 1, dim):  # 두 번째 변수 인덱스
                            for v3 in range(v2 + 1, dim):  # 세 번째 변수 인덱스
                                x_abc = basis(x[v1, :], j) * basis(x[v2, :], k) * basis(x[v3, :], m)
                                phi.append(x_abc)
                            
                                mapping = [0] * dim 
                                mapping[v1] = 1 
                                mapping[v2] = 1 
                                mapping[v3] = 1 
                                mapping_list.append(mapping)

    
    return np.array(phi).T, np.array(mapping_list).T

## 민감도 지수 산출과정


## 가우시안 분포에 대한 민감도 지수 함수(민감도 도합: 1)
# [검토 — 한현석, 2026-08-25] 미사용 경고:
#   이 함수는 기저가 "정규직교(orthonormal)"라서 계수 제곱합이 곧 분산이라는
#   가정 위에 있음. 이 노트북의 basis()는 정규화 안 된 표준 르장드르
#   다항식이라 이 가정이 성립하지 않음. 아래 코드에서는 실제로 이 함수를
#   호출하지 않고 get_sobol2(경험적 분산 기반)를 사용함.
#   정규직교화한 Hermite/Legendre 기저를 쓸 때만 이 함수를 사용할 것.
def get_sobol(Ci, mapping):
    
    total_var = np.sum(Ci[1:]**2)  # 전체 분산 (상수항 제외)
    sensitivity_dict = {}

    for i in range(1, Ci.shape[0]):
        # 현재 항에 관여된 변수 번호 추출 (예: [0, 2] -> X1, X3)
        active_indices = np.where(mapping[:, i] == 1)[0]
        
        # 'S' 뒤에 변수 번호를 붙여 키 생성 (예: S1, S13, S123)
        key = "S" + "".join(map(str, active_indices + 1))
        
        contribution = Ci[i]**2
        
        # 딕셔너리에 기여도 합산
        if key in sensitivity_dict:
            sensitivity_dict[key] += contribution
        else:
            sensitivity_dict[key] = contribution

    # 전체 분산으로 나누어 지수(Index)화
    for key in sensitivity_dict:
        sensitivity_dict[key] /= total_var
        
    return sensitivity_dict


## 르장드르 분포에 대한 민감도 지수 함수(민감도 도합: 1에 근사)
def get_sobol2(Ci, mapping, exp_input):
    sensitivity_dict = {}
    total_var_sum = 0.0 # 개별 분산들의 순수한 총합
    
    # 1. 상수항(i=0)을 제외한 각 항의 개별 기여도(분산) 계산 및 합산
    for i in range(1, Ci.shape[0]):
        active_indices = np.where(mapping[:, i] == 1)[0]
        if len(active_indices) == 0: continue
            
        key = "S" + "".join(map(str, sorted(active_indices + 1)))
        
        # np.var()는 무조건 양수(0 이상)만 반환.
        contribution = np.var(Ci[i] * exp_input[:, i])
        
        sensitivity_dict[key] = sensitivity_dict.get(key, 0) + contribution
        total_var_sum += contribution # 전체 분산 분모에 누적
        
    # 2. 비율 계산 (모든 항목이 양수이고, 자신의 합으로 나누므로 무조건 0 ~ 1 사이)
    for key in sensitivity_dict:
        if total_var_sum > 0:
            sensitivity_dict[key] /= total_var_sum
        else:
            sensitivity_dict[key] = 0
            
    return sensitivity_dict

#민감도 지수 나열
def display_detailed_sensitivity(sensitivity_dict):
    
    # 1. 딕셔너리 데이터를 리스트 형태로 변환 (데이터프레임 생성용)
    results = []
    for key, val in sensitivity_dict.items():
        results.append({'Interaction': key, 'Index': val})
    
    if not results:
        print("출력할 민감도 지수 데이터가 없습니다.")
        return
        
    df = pd.DataFrame(results)
    
    # 2. 정렬 및 순위 매기기
    df = df.sort_values(by='Index', ascending=False).reset_index(drop=True)
    # 순위 할당
    df['Rank'] = df['Index'].rank(ascending=False, method='min').astype(int)

    # 3. 표 출력
    print("\n" + "="*45)
    print(f"{'Interaction':^15} | {'Sensitivity Index':^18} | {'Rank':^8}")
    print("-" * 45)
    
    for _, row in df.iterrows():
        print(f"{row['Interaction']:^15} | {row['Index']:^18.6f} | {row['Rank']:^8}")
        
    print("-" * 45)
    # 모든 지수의 합계 출력 (이론적으로 1.0에 가까워야 함)
    total_sum = df['Index'].sum()
    print(f"{'Total Sum':^15} | {total_sum:^18.6f} |")
    print("="*45)


def find_optimal_degree(input_data, output_data, max_n=20, max_y=3,
                         val_ratio=0.2, patience=3, tol=1e-4, seed=0, verbose=True):
    # ---------------------------------------------------------
    # [수정 — 한현석, 2026-08-25]
    # 이유: 원래는 PDD 적합에 쓴 데이터로 그대로 R^2(adj R^2)를 계산해서
    #   차수를 골랐음. 항 개수(k)가 늘어날수록 학습 데이터 적합도는
    #   거의 항상 좋아지므로 이 방식은 사실상 가장 복잡한 모델을 고르는
    #   것과 같아 과적합. 게다가 목표치를 target_adj_r2=1.0으로 두면
    #   차수가 계속 올라가고(Ishigami 예제에서 실제로 n=18까지 상승),
    #   고차 다항식 기저는 조건수가 급격히 나빠져 수치적으로 불안정해짐

    # 변경 내용: 데이터를 학습/검증으로 분리하고, PDD 계수는 학습셋으로만
    #   추정한 뒤 검증셋 R^2"로 차수를 평가. 검증 R^2가 patience회
    #   연속으로 유의미하게(>tol) 개선되지 않으면 조기 종료.
    #   실제 카티아/CAE처럼 샘플 하나가 비싼 상황에서 과적합을 피하려면
    #   이 방식이 훨씬 안전함.
    # ---------------------------------------------------------
    rng = np.random.default_rng(seed)
    N_samples = input_data.shape[1]
    perm = rng.permutation(N_samples)
    n_val = max(int(N_samples * val_ratio), 1)
    val_idx, train_idx = perm[:n_val], perm[n_val:]

    X_train, X_val = input_data[:, train_idx], input_data[:, val_idx]
    Y_train, Y_val = output_data[train_idx], output_data[val_idx]

    best_n, best_y = 1, 1
    best_val_r2 = -float('inf')
    no_improve = 0

    for n in range(1, max_n + 1):
        for y in range(1, max_y + 1):
            # 1. PDD 행렬 생성 (학습셋 기준)
            exp_train, mapping = PDD(X_train, n, y)

            k = exp_train.shape[1] # 다항식 항의 총 개수

            # 학습 표본 수보다 항의 개수가 많아지면 건너뜀
            if k >= X_train.shape[1] - 1:
                continue

            # 2. 계수(Ci) 산출 (학습셋으로만)
            exp_li = np.linalg.pinv(exp_train)
            Ci = exp_li @ Y_train

            # 3. 검증셋에서 예측값 산출 및 R^2 계산
            exp_val, _ = PDD(X_val, n, y)
            predicted_val = exp_val @ Ci

            ss_res = np.sum((Y_val - predicted_val) ** 2)
            ss_tot = np.sum((Y_val - np.mean(Y_val)) ** 2)
            val_r2 = 1 - (ss_res / ss_tot)

            # 4. 최고 성능(검증 R^2 기준) 갱신
            if val_r2 > best_val_r2 + tol:
                best_val_r2 = val_r2
                best_n, best_y = n, y
                no_improve = 0
            else:
                no_improve += 1

            # ---------------------------------------------------------
            # 5. 조기 종료(Early Stopping): 검증 R^2가 patience회 연속 정체되면 종료
            if no_improve >= patience:
                if verbose:
                    print("-" * 55)
                    print(f"검증 R^2가 {patience}회 연속 유의미하게 개선되지 않아 탐색을 조기 종료합니다.")
                    print(f"최적의 조합: 차수 n={best_n}, 상호작용 y={best_y} (검증 R^2: {best_val_r2:.6f})")
                    print("="*55 + "\n")
                return best_n, best_y
            # ---------------------------------------------------------

    # 목표치에 도달하지 못하고 끝까지 탐색한 경우의 결과 출력
    if verbose:
        print("-" * 55)
        print(f"최적의 조합: 차수 n={best_n}, 상호작용 y={best_y} (검증 R^2: {best_val_r2:.6f})")
        print("="*55 + "\n")
    
    return best_n, best_y


exp_input, mapping = PDD(input,3,3)

exp_li = np.linalg.pinv(exp_input)

Ci = exp_li @ output

print(Ci.shape)

# 함수 실행
detailed_indices = get_sobol2(Ci, mapping, exp_input)

detailed_results = display_detailed_sensitivity(detailed_indices)


(10, 5000)
(5000,)
(286,)

  Interaction   | Sensitivity Index  |   Rank  
---------------------------------------------
     S568       |      0.037607      |    1    
      S59       |      0.036229      |    2    
     S379       |      0.034390      |    3    
     S410       |      0.033794      |    4    
      S16       |      0.030343      |    5    
     S279       |      0.030141      |    6    
     S6910      |      0.025566      |    7    
     S145       |      0.022792      |    8    
      S56       |      0.020028      |    9    
     S610       |      0.019862      |    10   
      S46       |      0.018791      |    11   
     S1710      |      0.018773      |    12   
      S67       |      0.018043      |    13   
      S5        |      0.017479      |    14   
      S28       |      0.017037      |    15   
     S349       |      0.016236      |    16   
     S124       |      0.014932      |    17   
      S47       |      0.014305      |    18   
      S25      

In [ ]:
import numpy as np

a = 7
b = 0.1

avg = a/2
var = 1/2 + (a**2)/8 + b*(np.pi)**4/5 + (b**2)*((np.pi)**8)/18

def Ishg(input):
    
    X1 = input[0, :]
    X2 = input[1, :]
    X3 = input[2, :]
    out = np.sin(X1) + a*(np.sin(X2))**2 + b*X3**4*np.sin(X1)
    
    return np.array(out)

input_Ishigami =  np.random.uniform(-np.pi, np.pi, (3, 100000))

output_Ishg = Ishg(input_Ishigami)

## 인풋을 스케일링 하여 PDD 함수 내에 대입
input_Ishigami = theoretical_scale(input_Ishigami, domain_min=-np.pi, domain_max=np.pi)

# [참고 — 한현석, 2026-08-25] find_optimal_degree가 검증셋 기반으로 바꿈.
# (위 셀 changelog 1번 참고)
opt_x, opt_y = find_optimal_degree(input_Ishigami, output_Ishg)

exp_input_Ishg, mapping_Ishg = PDD(input_Ishigami, opt_x, opt_y)

exp_li = np.linalg.pinv(exp_input_Ishg)

Ci_Ishg = exp_li @ output_Ishg



sobol_Ishg = get_sobol2(Ci_Ishg, mapping_Ishg, exp_input_Ishg)

Ishg_results = display_detailed_sensitivity(sobol_Ishg)

## 실제 분산과 민감도 지수
real_sobol = (((1+b*(np.pi)**4/5)**2)/2/var, a**2/8/var, 0, 8*b**2*np.pi**8/(225*var))
labels = ["S1", "S2", "S3", "S13"]

print("\n--- 이론적 민감도 지수 ---")
for label, value in zip(labels, real_sobol):
    print(f"{label:<5} : {value:.6f}")
print(f"{'합계':<5} : {sum(real_sobol):.6f}")
print("--------------------------")


-------------------------------------------------------
검증 R^2가 3회 연속 유의미하게 개선되지 않아 탐색을 조기 종료합니다.
최적의 조합: 차수 n=10, 상호작용 y=2 (검증 R^2: 0.999998)


  Interaction   | Sensitivity Index  |   Rank  
---------------------------------------------
      S2        |      0.441949      |    1    
      S1        |      0.313672      |    2    
      S13       |      0.244378      |    3    
      S23       |      0.000000      |    4    
      S12       |      0.000000      |    5    
      S3        |      0.000000      |    6    
---------------------------------------------
   Total Sum    |      1.000000      |

--- 이론적 민감도 지수 ---
S1    : 0.313905
S2    : 0.442411
S3    : 0.000000
S13   : 0.243684
합계    : 1.000000
--------------------------
